In [1]:
from __future__ import division
from __future__ import print_function
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.18.1
#   kernelspec:
#     display_name: Python 2
#     language: python
#     name: python2
# ---

import warnings
warnings.filterwarnings("ignore")

<h1>Outcomes Using Trust Features</h1>
Does performance improve for tasks (code status, leaving AMA, and in-hosp mortality) when adding mistrust features on top of demographics? 
Yes

In [2]:
from __future__ import absolute_import
from future import standard_library
from six.moves import map
from six.moves import range
from six.moves import zip
standard_library.install_aliases()
from builtins import zip
from builtins import map
from builtins import range
from past.utils import old_div
import psycopg2
import pandas as pd
from time import gmtime, strftime
import tqdm

In [3]:
# [repro] Compatibility wrapper for sklearn LogisticRegression

from sklearn.linear_model import LogisticRegression as _LR

class LogisticRegression(_LR):
    def __init__(
        self,
        penalty="l2",
        *,
        dual=False,
        tol=1e-4,
        C=1.0,
        fit_intercept=True,
        intercept_scaling=1,
        class_weight=None,
        random_state=None,
        solver="lbfgs",
        max_iter=100,
        multi_class="auto",
        verbose=0,
        warm_start=False,
        n_jobs=None,
        l1_ratio=None,
    ):
        # Inject compatibility defaults only when caller did not override them.
        if penalty == "l1":
            if solver == "lbfgs" or solver == "lbfgs":  # default for Python3
                solver = "liblinear"
            multi_class = "ovr"

        # Override tol for backwards-compatibility
        if tol == 1e-4:
            tol = 0.01

        # Override C default if matching old behaviour
        if C == 1.0:
            C = 0.1

        super().__init__(
            penalty=penalty,
            dual=dual,
            tol=tol,
            C=C,
            fit_intercept=fit_intercept,
            intercept_scaling=intercept_scaling,
            class_weight=class_weight,
            random_state=random_state,
            solver=solver,
            max_iter=max_iter,
            multi_class=multi_class,
            verbose=verbose,
            warm_start=warm_start,
            n_jobs=n_jobs,
            l1_ratio=l1_ratio,
        )

In [4]:
# con = psycopg2.connect(dbname ='mimic', user='wboag', host="/var/run/postgresql")
con = psycopg2.connect(dbname='mimic', user='wboag', host="host.docker.internal", port="5432")
cur = con.cursor()

In [5]:
print(strftime("%Y-%m-%d %H:%M:%S", gmtime()))

# LABEL: code status

code_query = "select distinct hadm_id,label,value from mimiciii.chartevents c JOIN mimiciii.d_items i on i.itemid=c.itemid where label = 'Code Status'"
code_status = pd.read_sql_query(code_query, con)

# binary labels
code_labels = {}
for i,row in tqdm.tqdm(code_status.iterrows()):
    if row.value is not None:
        if ('DNR' in row.value) or ('DNI' in row.value) or ('Comfort' in row.value) or ('Do Not' in row.value):
            label = 'DNR/CMO'
        elif (row.value == 'Full Code') or (row.value == 'Full code'):
            label = 'Full Code'
    code_labels[row.hadm_id] = label
    
#ode_status.head()

2025-12-02 20:38:38


46121it [00:02, 22958.41it/s]


In [6]:
print(set(code_status['value'].values))

{'DNR (do not resuscitate)', 'Full code', 'Do Not Intubate', None, 'DNR / DNI', 'Full Code', 'CPR Not Indicate', 'Other/Remarks', 'Do Not Resuscita', 'DNI (do not intubate)', 'Comfort measures only', 'Comfort Measures'}


In [7]:
# hadm -> race
import tqdm

def normalize_race(race):
    if 'HISPANIC' in race:
        return 'Hispanic'
    if 'SOUTH AMERICAN' in race:
        return 'Hispanic'
    if 'AMERICAN INDIAN' in race:
        return 'Native American'
    if 'ASIAN' in race:
        return 'Asian'
    if 'BLACK' in race:
        return 'Black'
    if 'WHITE' in race:
        return 'White'
    return 'Other'

def normalize_insurance(insurance):
    if insurance in ['Medicare', 'Medicaid', 'Government']:
        return 'Public'
    else:
        return insurance

In [8]:
# LABEL: left hospital against medical advice

# query for discharge info
discharge_query = 'SELECT distinct hadm_id,discharge_location FROM mimiciii.admissions'
discharge = pd.read_sql_query(discharge_query, con)

# binary labels
ama_labels = {}
for i,row in tqdm.tqdm(discharge.iterrows()):
    if row.discharge_location == 'LEFT AGAINST MEDICAL ADVI':
        label = 'AMA'
    else:
        label = 'compliant'
    ama_labels[row.hadm_id] = label

#discharge.head()

58976it [00:01, 31807.48it/s]


In [9]:
# LABEL: in-hospital mortality

# query for discharge info
mortality_query = 'SELECT distinct hadm_id,hospital_expire_flag FROM mimiciii.admissions'
mortality = pd.read_sql_query(mortality_query, con)

# binary labels
mortality_labels = {}
for i,row in tqdm.tqdm(mortality.iterrows()):
    if row.hospital_expire_flag:
        label = 'deceased'
    else:
        label = 'survived'
    mortality_labels[row.hadm_id] = label

#mortality.head()

58976it [00:01, 33346.66it/s]


In [10]:
import random

def data_split(ids, ratio=0.6):
    random.shuffle(ids)
    train = ids[:int(len(ids)*ratio) ]
    test  = ids[ int(len(ids)*ratio):]
    return train, test

In [11]:
# write informative features code

def analyze(task, vect, clf, count_top=False):

    ind2feat =  { i:f for f,i in list(vect.vocabulary_.items()) }

    # create a 2-by-m matrix for biary, rather than relying on 1-p bullshit
    coef_ = clf.coef_
    
    # most informative features
    #"""
    print(task)
    informative_feats = np.argsort(coef_)
    
    if len(informative_feats.shape) == 2:
        informative_feats = informative_feats[0,:]
        coef_ = coef_[0,:]
        
    #'''
    # display what each feature is
    for feat in reversed(informative_feats):
        val = coef_[feat]

        word = ind2feat[feat]
        print('\t%-25s: %7.4f' % (word,val))
        


In [12]:
%matplotlib inline

import numpy as np
import sklearn
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt
import pylab as plt


def compute_stats(task, pred, P, ref, labels_map, verbose):
    if len(labels_map) == 2:
        scores = P[:,1] - P[:,0]
        res = compute_stats_binary(    task, pred, scores, ref, labels_map, verbose)
    else:
        res = compute_stats_multiclass(task, pred, P     , ref, labels_map, verbose)
    return res



def compute_stats_binary(task, pred, P, ref, labels, verbose):
    # santiy check
    assert all(list(map(int,P>0)) == pred)

    V = [0,1]
    n = len(V)
    assert n==2, 'sorry, must be exactly two labels (how else would we do AUC?)'
    conf = np.zeros((n,n), dtype='int32')
    for p,r in zip(pred,ref):
        conf[p][r] += 1

    if verbose:
        print(conf)
        print()
    
    tp = conf[1,1]
    tn = conf[0,0]
    fp = conf[1,0]
    fn = conf[0,1]

    precision   = old_div(tp, (tp + fp + 1e-9))
    recall      = old_div(tp, (tp + fn + 1e-9))
    sensitivity = old_div(tp, (tp + fn + 1e-9))
    specificity = old_div(tn, (tn + fp + 1e-9))

    f1 = old_div((2*precision*recall), (precision+recall+1e-9))

    tpr =  true_positive_rate(pred, ref)
    fpr = false_positive_rate(pred, ref)

    accuracy = old_div((tp+tn), (tp+tn+fp+fn + 1e-9))
    
    if verbose:
        print('\tspecificity %.3f' % specificity)
        print('\tsensitivty: %.3f' % sensitivity)

    # AUC
    if len(set(ref)) == 2:
        auc = sklearn.metrics.roc_auc_score(ref, P)
        if verbose: print('\t\tauc:        %.3f' % auc)

    if verbose:
        print('\taccuracy:   %.3f' % accuracy)
        print('\tprecision:  %.3f' % precision)
        print('\trecall:     %.3f' % recall)
        print('\tf1:         %.3f' % f1)
        print('\tTPR:        %.3f' % tpr)
        print('\tFPR:        %.3f' % fpr)

        print('TODO: VIZ THE ROC CURVE')

    res = {'accuracy':accuracy, 'precision':precision, 'recall':recall, 'f1':f1, 'tpr':tpr,
           'fpr':fpr, 'auc':auc, 'sensitivity':sensitivity, 'specificity':specificity}

    return res



def compute_stats_multiclass(task, pred, P, ref, labels_map):
    # santiy check
    assert all(list(map(int,P.argmax(axis=1))) == pred)

    # get rid of that final prediction dimension
    #pred = pred[1:]
    #ref  =  ref[1:]

    V = set(range(len(labels_map)))
    n = max(V)+1
    conf = np.zeros((n,n), dtype='int32')
    for p,r in zip(pred,ref):
        conf[p][r] += 1


    labels = [label for label,i in sorted(list(labels_map.items()), key=lambda t:t[1])]


    print(conf)
    print()
    
    precisions = []
    recalls = []
    f1s = []
    print('\t prec  rec    f1   label')
    for i in range(n):
        label = labels[i]

        tp = conf[i,i]
        pred_pos = conf[i,:].sum()
        ref_pos  = conf[:,i].sum()

        precision   = old_div(tp, (pred_pos + 1e-9))
        recall      = old_div(tp, (ref_pos + 1e-9))
        f1 = old_div((2*precision*recall), (precision+recall+1e-9))

        print('\t%.3f %.3f %.3f %s' % (precision,recall,f1,label))

        # Save info
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    avg_precision = old_div(sum(precisions), len(precisions))
    avg_recall    = old_div(sum(recalls   ), len(recalls   ))
    avg_f1        = old_div(sum(f1s       ), len(f1s       ))
    print('\t--------------------------')
    print('\t%.3f %.3f %.3f avg' % (avg_precision,avg_recall,avg_f1))

    print('TODO: VIZ THE F1S')

    
    res = {'precisions':precisions, 'recalls':recalls, 'f1s':f1s}

    return res



def true_positive_rate(pred, ref):
    tp,fn = 0,0
    for p,r in zip(pred,ref):
        if p==1 and r==1:
            tp += 1
        elif p==0 and r==1:
            fn += 1
    return old_div(tp, (tp + fn + 1e-9))


def false_positive_rate(pred, ref):
    fp,tn = 0,0
    for p,r in zip(pred,ref):
        if p==1 and r==0:
            fp += 1
        elif p==0 and r==0:
            tn += 1
    return old_div(fp, (fp + tn + 1e-9))




def classification_results(svm, labels_map, X, Y, task, verbose=True):

    # for AUC
    P_ = svm.decision_function(X)

    # sklearn has stupid-ass changes in API when doing binary classification. make it conform to 3+
    if len(labels_map)==2:
        m = X.shape[0]
        P = np.zeros((m,2))
        P[:,0] = -P_
        P[:,1] =  P_
    else:
        P = P_

    train_pred = P.argmax(axis=1)

    # what is the predicted vocab without the dummy label?
    V = list(labels_map.keys())

    if verbose: print(task)
    res = compute_stats(task, train_pred, P, Y, labels_map, verbose)
    if verbose: print('\n')
    return res
    


def regression_results(lr, test_X, test_Y, description, verbose=True):
    res = {}
    
    pred_Y = lr.predict(test_X)
    res['rms'] = sqrt(mean_squared_error(test_Y, pred_Y))
    res['mas'] = mean_absolute_error(test_Y, pred_Y)
    if verbose:
        print(description)
        print('\tRMS:', res['rms'])
        print('\tMAS:', res['mas'])
        print()
    
        fig = plt.figure()
        perfect = np.arange(min(test_Y),max(test_Y),100)
        plt.scatter(perfect, perfect, color='red', s=0.01)
        plt.scatter(test_Y , pred_Y, color='blue', s=1)
        plt.xlabel('actual')
        plt.ylabel('prediction')
        plt.show()
    
    return res

In [13]:
# Load features

import pickle as pickle

def normalize(scores):
    vals = np.array(list(scores.values()))
    mu = vals.mean()
    std = vals.std()
    return { k:old_div((v-mu),std) for k,v in list(scores.items())}


# query for insurance info
insurance_query = 'SELECT distinct hadm_id,insurance FROM mimiciii.admissions'
insurance = pd.read_sql_query(insurance_query, con)

# query for oasis info
oasis_query = 'SELECT distinct hadm_id,oasis FROM mimiciii.oasis'
oasis = pd.read_sql_query(oasis_query, con)

# [repro] note: updated age to admission_age
# [repro] note: admission_type not present, omitted
# query for demographics info
patients_query = 'SELECT distinct hadm_id,gender,admission_age as age,ethnicity,los_hospital FROM mimiciii.icustay_detail WHERE first_icu_stay = True;'
patients = pd.read_sql_query(patients_query, con)
# patients = patients.loc[patients['admission_type']!='NEWBORN']

# Load trust scores
with open('../data/mistrust_noncompliant.pkl', 'rb') as f:
    noncompliant_dict = normalize(pickle.load(f))
print('noncompliant:', len(noncompliant_dict))
noncompliant_df = pd.DataFrame(list(noncompliant_dict.items()), columns=['hadm_id','noncompliant'])

# Load trust scores
with open('../data/mistrust_autopsy.pkl', 'rb') as f:
    autopsy_dict = normalize(pickle.load(f))
print('autopsy:', len(autopsy_dict))
autopsy_df = pd.DataFrame(list(autopsy_dict.items()), columns=['hadm_id','autopsy'])

# Load trust scores
with open('../data/neg_sentiment.pkl', 'rb') as f:
    sentiment_dict = normalize(pickle.load(f))
print('sentiment:', len(sentiment_dict))
sentiment_df = pd.DataFrame(list(sentiment_dict.items()), columns=['hadm_id','sentiment'])

    
# merge data
extra_1 = pd.merge(insurance, oasis, on=['hadm_id'])
extra_2 = pd.merge(extra_1, noncompliant_df, on=['hadm_id'])
extra_3 = pd.merge(extra_2, autopsy_df     , on=['hadm_id'])
extra_4 = pd.merge(extra_3, sentiment_df   , on=['hadm_id'])
demographics = pd.merge(extra_4, patients  , on=['hadm_id'])

# Normalize some columns
demographics['ethnicity'] = demographics['ethnicity'].apply(normalize_race)
demographics['insurance'] = demographics['insurance'].apply(normalize_insurance)
demographics = demographics.rename(columns={'ethnicity':'race'})
demographics = demographics.rename(columns={'los_hospital':'los'})

#demographics.head()

noncompliant: 54510
autopsy: 54510
sentiment: 52726


In [14]:
import numpy as np
from sklearn.feature_extraction import DictVectorizer

print(strftime("%Y-%m-%d %H:%M:%S"))


def normalize_mean_std(value, mu, std):
    return old_div((value-mu),std)
    
# normalize ages
ages = np.array(demographics['age'])
age_mu = ages.mean()
age_std = ages.std()
demographics['age'] = demographics['age'].apply(lambda val:normalize_mean_std(val,age_mu,age_std))

# normalize oasis scores
oasis = np.array(demographics['oasis'])
oasis_mu = oasis.mean()
oasis_std = oasis.std()
demographics['oasis'] = demographics['oasis'].apply(lambda val:normalize_mean_std(val,oasis_mu,oasis_std))

# normalize los scores
los = np.array(demographics['los'])
los_mu = los.mean()
los_std = los.std()
demographics['los'] = demographics['los'].apply(lambda val:normalize_mean_std(val,los_mu,los_std))

# foo

def build_features(enabled):
    demographics_features = {}
    for i,row in tqdm.tqdm(demographics.iterrows()):
        feats = {}

        # if 'admission_type' in enabled: feats[('admission_type', row.admission_type   )] = 1
        if 'oasis'          in enabled: feats[('oasis', None)] = row.oasis

        if 'age' in enabled: feats[('age'  , None)] = row.age
        if 'los' in enabled: feats[('los'  , None)] = row.los

        if 'insurance' in enabled: feats[('insurance'     , row.insurance)] = 1
        if 'gender'    in enabled: feats[('gender'        , row.gender   )] = 1

        if 'race'     in enabled: feats[('race', row.race     )] = 1
            
        if 'noncompliant' in enabled: feats[('concompliant',None)] = row.noncompliant
        if 'autopsy'      in enabled: feats[('autopsy'     ,None)] = row.autopsy
        if 'sentiment'    in enabled: feats[('sentiment'   ,None)] = row.sentiment

        demographics_features[row.hadm_id] = feats

    print(strftime("%Y-%m-%d %H:%M:%S"))

    # fit vectorizer
    vect = DictVectorizer()
    vect.fit(list(demographics_features.values()))
    print('num_features:', len(vect.get_feature_names()))

    # ordering of all features
    ids = list(demographics_features.keys())
    print('\t', strftime("%Y-%m-%d %H:%M:%S"))
    X = vect.transform([demographics_features[hadm_id] for hadm_id in ids])    

    return demographics_features, vect
    
print(strftime("%Y-%m-%d %H:%M:%S"))

2025-12-02 20:38:46
2025-12-02 20:38:46


In [15]:
# AMA
from collections import defaultdict, Counter

print(strftime("%Y-%m-%d %H:%M:%S"))
from sklearn.linear_model import LogisticRegression



featlists = {
                #'BASELINE'             :['age', 'los', 'insurance', 'gender'],
                #'BASELINE+RACE'        :['age', 'los', 'insurance', 'gender', 'race'],
                #'BASELINE+NONCOMPLIANT':['age', 'los', 'insurance', 'gender', 'noncompliant'],
                #'BASELINE+AUTOPSY'     :['age', 'los', 'insurance', 'gender', 'autopsy'],
                #'BASELINE+SENTIMENT'   :['age', 'los', 'insurance', 'gender', 'sentiment'],
                'BASELINE+ALL'         :['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']
            }

ama_Y_vect = {'AMA': 1, 'compliant': 0}

feature_weights = defaultdict(list)

for name,featlist in list(featlists.items()):
    print(name)
    print(featlist)
    
    demographics_features, vect = build_features(featlist)
    ind2feat =  { i:f for f,i in list(vect.vocabulary_.items()) }

    ama_ids = list(set(discharge['hadm_id'].values) & set(demographics_features.keys()))
    print('patients:', len(ama_ids))
  
    print(Counter([ama_Y_vect[ama_labels[hadm_id]] for hadm_id in ama_ids]))

    aucs = []
    for iteration in tqdm.tqdm(list(range(100))):

        # train/test split
        ama_train_ids, ama_test_ids = data_split(ama_ids)

        # select pre-computed features
        ama_train_features = [demographics_features[hadm_id] for hadm_id in ama_train_ids]
        ama_test_features  = [demographics_features[hadm_id] for hadm_id in ama_test_ids ]

        # vectorize features
        ama_train_X = vect.transform(ama_train_features)
        ama_test_X  = vect.transform(ama_test_features)

        # vectorize task-specific labels
        #print ama_Y_vect

        # select labels
        ama_train_Y = [ama_Y_vect[ama_labels[hadm_id]] for hadm_id in ama_train_ids]
        ama_test_Y  = [ama_Y_vect[ama_labels[hadm_id]] for hadm_id in ama_test_ids ]

        # fit model
        # [repro] set solver to 'liblinear' for l1 penalty
        ama_svm = LogisticRegression(C=0.1, penalty='l1', tol=0.01, solver='liblinear')
        ama_svm.fit(ama_train_X,ama_train_Y)
        #print ama_svm


        # AMA Model eval

        # evaluate model
        res = classification_results(ama_svm, ama_Y_vect,  ama_test_X,  ama_test_Y, 'test:  ama', verbose=False)
        aucs.append(res['auc'])

        # record the weights of the features (because we average them)
        if name == 'BASELINE+ALL':
            for feat,val in enumerate(ama_svm.coef_.tolist()[0]):
                featname = ind2feat[feat]
                feature_weights[featname].append(val)

        #classification_results(ama_svm, ama_Y_vect, ama_train_X, ama_train_Y, 'train: ama')

    aucs = np.array(aucs)
    print('AUCS: ', aucs)
    print('    mean:     ', aucs.mean())
    print('    1.96*std: ', aucs.std() * 1.96)
    print('    conf_interval: (%.4f,%.4f)' % (aucs.mean()-1.96*aucs.std(),aucs.mean()+1.96*aucs.std()))


    # most informative features
    analyze('ama', vect, ama_svm)
    print('\n\n\n')

    if name == 'BASELINE+ALL':
        for featname,vals in sorted(feature_weights.items()):
            v = np.array(vals)
            mu = v.mean()
            std = v.std()
            print('%-10s:%-15s || %.2f +/- %.2f' % (featname[0],featname[1],mu,1.96*std))

print(strftime("%Y-%m-%d %H:%M:%S"))

2025-12-02 20:38:46
BASELINE+ALL
['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']


54566it [00:02, 19732.63it/s]


2025-12-02 20:38:49
num_features: 16
	 2025-12-02 20:38:49
patients: 51089
Counter({0: 50789, 1: 300})


100%|██████████| 100/100 [00:35<00:00,  2.83it/s]

AUCS:  [0.8782944  0.86684197 0.876739   0.87965502 0.8630978  0.87542947
 0.86469683 0.86909279 0.84629056 0.88352836 0.87557967 0.86025733
 0.87768124 0.8618229  0.88055261 0.86563939 0.8710702  0.89219374
 0.86707693 0.87994592 0.87465159 0.8700914  0.87456438 0.86921352
 0.88167007 0.89291381 0.88041362 0.85588979 0.87651898 0.86914735
 0.8861094  0.90602361 0.85604607 0.85875556 0.87312961 0.84159672
 0.88990481 0.87294749 0.86588378 0.86953633 0.88971081 0.91716814
 0.87975285 0.87014205 0.85975238 0.86371171 0.9088993  0.87794698
 0.88613369 0.86618061 0.88525803 0.8881934  0.86470426 0.88180014
 0.8676546  0.87260671 0.86367682 0.85917457 0.88339186 0.85992174
 0.86021915 0.85925778 0.82678173 0.87939891 0.88900825 0.88799495
 0.86531187 0.87296763 0.89066011 0.87090772 0.84441448 0.85595165
 0.86150859 0.88925065 0.88138484 0.86493002 0.87795059 0.88026735
 0.8766544  0.87191893 0.88122436 0.87444257 0.89338042 0.89017725
 0.86743937 0.88877495 0.88795014 0.88077536 0.86952088

In [16]:
# Code Status

print(strftime("%Y-%m-%d %H:%M:%S"))


from sklearn.linear_model import LogisticRegression



featlists = {
                #'BASELINE'             :['age', 'los', 'insurance', 'gender'],
                #'BASELINE+RACE'        :['age', 'los', 'insurance', 'gender', 'race'],
                #'BASELINE+NONCOMPLIANT':['age', 'los', 'insurance', 'gender', 'noncompliant'],
                #'BASELINE+AUTOPSY'     :['age', 'los', 'insurance', 'gender', 'autopsy'],
                #'BASELINE+SENTIMENT'   :['age', 'los', 'insurance', 'gender', 'sentiment'],
                'BASELINE+ALL'         :['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']
            }

cs_Y_vect = {'DNR/CMO': 1, 'Full Code': 0}

feature_weights = defaultdict(list)

for name,featlist in list(featlists.items()):
    print(name)
    print(featlist)
    
    demographics_features, vect = build_features(featlist)
    ind2feat =  { i:f for f,i in list(vect.vocabulary_.items()) }
    
    print(ind2feat)

    cs_ids = list(set(code_labels.keys()) & set(demographics_features.keys()))
    print('patients:', len(cs_ids))
    
    print(Counter([cs_Y_vect[code_labels[hadm_id]] for hadm_id in cs_ids]))

    
    aucs = []
    for iteration in tqdm.tqdm(list(range(100))):

        #print 'Iter:', iteration

        # train/test split
        cs_train_ids, cs_test_ids = data_split(cs_ids)

        # select pre-computed features
        cs_train_features = [demographics_features[hadm_id] for hadm_id in cs_train_ids]
        cs_test_features  = [demographics_features[hadm_id] for hadm_id in cs_test_ids ]

        # vectorize features
        cs_train_X = vect.transform(cs_train_features)
        cs_test_X  = vect.transform(cs_test_features)

        # vectorize task-specific labels
        cs_Y_vect = {'DNR/CMO': 1, 'Full Code': 0}
        #print cs_Y_vect

        # select labels
        cs_train_Y = [cs_Y_vect[code_labels[hadm_id]] for hadm_id in cs_train_ids]
        cs_test_Y  = [cs_Y_vect[code_labels[hadm_id]] for hadm_id in cs_test_ids ]

        # fit model
        # [repro] set solver to 'liblinear' for l1 penalty
        cs_svm = LogisticRegression(C=0.1, penalty='l1', tol=0.01, solver='liblinear')
        cs_svm.fit(cs_train_X,cs_train_Y)
        #print cs_svm


        # cs Model eval

        # evaluate model
        res = classification_results(cs_svm, cs_Y_vect, cs_test_X,  cs_test_Y, 'test:  cs', verbose=False)
        aucs.append(res['auc'])

        # record the weights of the features (because we average them)
        if name == 'BASELINE+ALL':
            for feat,val in enumerate(cs_svm.coef_.tolist()[0]):
                featname = ind2feat[feat]
                feature_weights[featname].append(val)
            
        # most informative features
        #analyze('cs', vect, cs_svm)
        
    aucs = np.array(aucs)
    print('AUCS: ', aucs)
    print('    mean:     ', aucs.mean())
    print('    1.96*std: ', aucs.std() * 1.96)
    print('    conf_interval: (%.4f,%.4f)' % (aucs.mean()-1.96*aucs.std(),aucs.mean()+1.96*aucs.std()))


    # most informative features
    analyze('cs', vect, cs_svm)

    if name == 'BASELINE+ALL':
        for featname,vals in sorted(feature_weights.items()):
            v = np.array(vals)
            mu = v.mean()
            std = v.std()
            print('%-10s:%-15s || %.2f +/- %.2f' % (featname[0],featname[1],mu,1.96*std))
    
print(strftime("%Y-%m-%d %H:%M:%S"))

2025-12-02 20:39:25
BASELINE+ALL
['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']


54566it [00:02, 19919.94it/s]


2025-12-02 20:39:27
num_features: 16
	 2025-12-02 20:39:28
{0: ('age', None), 1: ('autopsy', None), 2: ('concompliant', None), 3: ('gender', 'F'), 4: ('gender', 'M'), 5: ('insurance', 'Private'), 6: ('insurance', 'Public'), 7: ('insurance', 'Self Pay'), 8: ('los', None), 9: ('race', 'Asian'), 10: ('race', 'Black'), 11: ('race', 'Hispanic'), 12: ('race', 'Native American'), 13: ('race', 'Other'), 14: ('race', 'White'), 15: ('sentiment', None)}
patients: 39125
Counter({0: 36667, 1: 2458})


100%|██████████| 100/100 [00:29<00:00,  3.38it/s]

AUCS:  [0.7802709  0.78817358 0.78246094 0.78507689 0.78855949 0.78878
 0.78268149 0.78628038 0.78576261 0.78497242 0.7753103  0.78469927
 0.78926779 0.77732877 0.78365931 0.78041035 0.78203282 0.78822852
 0.78084944 0.79204669 0.77789478 0.79364715 0.77888775 0.78458337
 0.78489041 0.7828733  0.78550789 0.77789128 0.78839454 0.77773607
 0.78187035 0.78405362 0.77891992 0.78731563 0.77221527 0.79000662
 0.79715057 0.78205841 0.78978874 0.78814126 0.78568699 0.79463749
 0.78338963 0.79257851 0.77985878 0.78562303 0.77524485 0.78703924
 0.79156027 0.78283967 0.77920328 0.78977596 0.78092955 0.79256011
 0.77850099 0.77836809 0.78646489 0.77722505 0.77691787 0.78210799
 0.79527539 0.78980624 0.78606123 0.78193199 0.788527   0.77314979
 0.78529764 0.7799171  0.79217737 0.79486379 0.77790644 0.78267921
 0.78143285 0.78053607 0.77940137 0.79416628 0.78283011 0.79105874
 0.78663164 0.77268333 0.7834803  0.78684    0.7880029  0.78154527
 0.78532625 0.78721332 0.78870241 0.78307359 0.77949912 0.

In [17]:
# Mortality

print(strftime("%Y-%m-%d %H:%M:%S"))


from sklearn.linear_model import LogisticRegression


featlists = {
                #'BASELINE'             :['age', 'los', 'insurance', 'gender'],
                #'BASELINE+RACE'        :['age', 'los', 'insurance', 'gender', 'race'],
                #'BASELINE+NONCOMPLIANT':['age', 'los', 'insurance', 'gender', 'noncompliant'],
                #'BASELINE+AUTOPSY'     :['age', 'los', 'insurance', 'gender', 'autopsy'],
                #'BASELINE+SENTIMENT'   :['age', 'los', 'insurance', 'gender', 'sentiment'],
                'BASELINE+ALL'         :['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']
            }
mortality_Y_vect = {'deceased': 1, 'survived': 0}

feature_weights = defaultdict(list)


for name,featlist in list(featlists.items()):
    print(name)
    print(featlist)
    
    demographics_features, vect = build_features(featlist)
    ind2feat =  { i:f for f,i in list(vect.vocabulary_.items()) }

    mortality_ids = list(set(mortality_labels.keys()) & set(demographics_features.keys()))
    print('patients:', len(mortality_ids))
    
    print(Counter([mortality_Y_vect[mortality_labels[hadm_id]] for hadm_id in mortality_ids]))

    aucs = []
    for iteration in tqdm.tqdm(list(range(100))):

        #print 'Iter:', iteration


        # train/test split
        mortality_train_ids, mortality_test_ids = data_split(mortality_ids)

        # select pre-computed features
        mortality_train_features = [demographics_features[hadm_id] for hadm_id in mortality_train_ids]
        mortality_test_features  = [demographics_features[hadm_id] for hadm_id in mortality_test_ids ]

        # vectorize features
        mortality_train_X = vect.transform(mortality_train_features)
        mortality_test_X  = vect.transform(mortality_test_features)

        # vectorize task-specific labels
        #print mortality_Y_vect

        # select labels
        mortality_train_Y = [mortality_Y_vect[mortality_labels[hadm_id]] for hadm_id in mortality_train_ids]
        mortality_test_Y  = [mortality_Y_vect[mortality_labels[hadm_id]] for hadm_id in mortality_test_ids ]

        # fit model
        # [repro] set solver to 'liblinear' for l1 penalty
        mortality_svm = LogisticRegression(C=0.1, penalty='l1', tol=0.01, solver='liblinear')
        mortality_svm.fit(mortality_train_X,mortality_train_Y)
        #print mortality_svm


        # mortality Model eval

        # evaluate model
        res = classification_results(mortality_svm, mortality_Y_vect, mortality_test_X, mortality_test_Y, 'test:  mortality', verbose=False)
        aucs.append(res['auc'])

        # record the weights of the features (because we average them)
        if name == 'BASELINE+ALL':
            for feat,val in enumerate(mortality_svm.coef_.tolist()[0]):
                featname = ind2feat[feat]
                feature_weights[featname].append(val)
            
        # most informative features
        #analyze('mortality', vect, mortality_svm)
        
    aucs = np.array(aucs)
    print('AUCS: ', aucs)
    print('    mean:     ', aucs.mean())
    print('    1.96*std: ', aucs.std() * 1.96)
    print('    conf_interval: (%.4f,%.4f)' % (aucs.mean()-1.96*aucs.std(),aucs.mean()+1.96*aucs.std()))


    # most informative features
    analyze('mortality', vect, mortality_svm)

    # foo


    if name == 'BASELINE+ALL':
        for featname,vals in sorted(feature_weights.items()):
            v = np.array(vals)
            mu = v.mean()
            std = v.std()
            print('%-10s:%-15s || %.2f +/- %.2f' % (featname[0],featname[1],mu,1.96*std))
            
    print('\n\n')
    
print(strftime("%Y-%m-%d %H:%M:%S"))

2025-12-02 20:39:57
BASELINE+ALL
['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']


54566it [00:02, 19410.01it/s]


2025-12-02 20:40:00
num_features: 16
	 2025-12-02 20:40:00
patients: 51089
Counter({0: 45921, 1: 5168})


100%|██████████| 100/100 [00:34<00:00,  2.88it/s]

AUCS:  [0.69322722 0.68807323 0.69177326 0.68851042 0.69049081 0.68483373
 0.69394479 0.69808311 0.68890749 0.68891775 0.69711962 0.69005268
 0.69142481 0.69975247 0.69440116 0.68583128 0.69033367 0.69608895
 0.69172524 0.68757854 0.69266888 0.6920985  0.69052432 0.69065229
 0.68638998 0.68987448 0.69687541 0.70034298 0.69316709 0.68866904
 0.69213361 0.69153967 0.69681295 0.7051373  0.69740141 0.69064605
 0.69794986 0.69974475 0.68971819 0.69223779 0.68659887 0.68999952
 0.6974492  0.69343079 0.69154581 0.69167303 0.69661565 0.69173225
 0.70102951 0.69378414 0.69378027 0.68859758 0.68959443 0.69065532
 0.69021297 0.69726291 0.69031443 0.69606646 0.69510403 0.68860157
 0.68813426 0.69210425 0.69318199 0.69587111 0.70191052 0.69050966
 0.69443308 0.68834616 0.68432802 0.68477911 0.69796992 0.69402125
 0.69340243 0.6966815  0.69550241 0.70493156 0.69046015 0.68939734
 0.69278884 0.70156237 0.70344147 0.68630694 0.69754371 0.69515809
 0.69652502 0.6974596  0.69473551 0.69837439 0.70085651

In [18]:

metrics = {'noncompliant':noncompliant_dict, 'autopsy':autopsy_dict, 'sentiment':sentiment_dict}

for metric,scores in list(metrics.items()):
    print(metric)

    vals = sorted(scores.values())
    n = len(vals)
    t1 = vals[old_div(1*n,4)]
    t2 = vals[old_div(2*n,4)]
    t3 = vals[old_div(3*n,4)]

    lowest  = [hadm_id for hadm_id,score in list(scores.items()) if     score<=t1]
    highest = [hadm_id for hadm_id,score in list(scores.items()) if t3< score    ]

    def mort_rate(label, hadm_ids):
        cohort = mortality.loc[mortality['hadm_id'].isin(hadm_ids)]
        print('\t', label, sum(cohort['hospital_expire_flag'].values)/float(len(cohort)))

    mort_rate('most  trust', lowest)
    mort_rate('least trust', highest)

noncompliant
	 most  trust 0.03888766600631007
	 least trust 0.13572267920094008
autopsy
	 most  trust 0.11706776234265963
	 least trust 0.08657403601085259
sentiment
	 most  trust 0.04490972538309816
	 least trust 0.16614824368409073
